# Notebook 1: 数据准备与探索

本 Notebook 完成以下工作:
1. 从 HuggingFace 下载 Jackrong 蒸馏数据集
2. 数据探索与统计分析
3. 数据清洗与格式校验
4. 划分训练/验证集
5. 导出为 ChatML 格式，供 SFT 训练使用
6. 过滤高质量样本，生成 RL 训练数据 (prompt+answer 对)

**数据集来源** (Apache 2.0 License):
- Chinese-Qwen3-235B-Thinking-2507-Distill-100k
- gpt-oss-120b-distilled-reasoning

**运行环境**: 4× A5000 (24GB each), Python 3.10+
**预计耗时**: ~30 分钟 (含数据下载)

In [ ]:
# ============================================================
# 0. 环境初始化
# ============================================================

import os
import sys
import json
import random
from pathlib import Path
from collections import Counter

import pandas as pd
import numpy as np
from datasets import load_dataset, concatenate_datasets, Dataset, DatasetDict
from tqdm.auto import tqdm

# 添加项目根目录到 path
PROJECT_ROOT = Path(os.getcwd()).parent if Path(os.getcwd()).name == 'notebooks' else Path(os.getcwd())
sys.path.insert(0, str(PROJECT_ROOT))

from utils.schema import SFTDataFormatter, RLDataFormatter, TrainingSample

# 随机种子
SEED = 3407
random.seed(SEED)
np.random.seed(SEED)

print(f" Project Root: {PROJECT_ROOT}")
print(f" Python: {sys.version}")

In [ ]:
# ============================================================
# 1. 下载数据集
# ============================================================

# Jackrong 蒸馏数据集 (Apache 2.0)
DATASETS = [
    "R6410418/Chinese-Qwen3-235B-Thinking-2507-Distill-100k",
    "R6410418/gpt-oss-120b-distilled-reasoning",
]

print(" Loading datasets from HuggingFace...")
all_datasets = []

for ds_name in DATASETS:
    print(f"\n  Downloading: {ds_name}")
    try:
        ds = load_dataset(ds_name, split="train")
        print(f"    ✅ Loaded: {len(ds)} samples")
        print(f"    Features: {ds.features}")
        all_datasets.append(ds)
    except Exception as e:
        print(f"    ❌ Failed: {e}")

# 合并所有数据集
if all_datasets:
    raw_dataset = concatenate_datasets(all_datasets)
    print(f"\n Total samples before filtering: {len(raw_dataset)}")
else:
    print("\n ❌ No datasets loaded! Check network or dataset names.")
    # fallback: 创建 demo 数据
    raw_dataset = Dataset.from_dict({"messages": []})

In [ ]:
# ============================================================
# 2. 数据探索
# ============================================================

# 查看样本
print("=== Sample 0 ===")
sample = raw_dataset[0]
for msg in sample.get("messages", []):
    role = msg.get("role", "unknown")
    content = msg.get("content", "")[:150]
    print(f"  [{role}] {content}...")

# 统计信息
print(f"\n=== Dataset Statistics ===")
print(f"Total samples: {len(raw_dataset)}")
print(f"Features: {raw_dataset.features}")

# 统计对话轮数分布
turn_counts = []
content_lengths = []
has_system = 0
has_assistant = 0

for sample in tqdm(raw_dataset, desc="Analyzing"):
    msgs = sample.get("messages", [])
    turn_counts.append(len(msgs))
    for msg in msgs:
        content_lengths.append(len(msg.get("content", "")))
        if msg.get("role") == "system":
            has_system += 1
        if msg.get("role") == "assistant":
            has_assistant += 1

print(f"\nAvg turns: {np.mean(turn_counts):.1f} ± {np.std(turn_counts):.1f}")
print(f"Avg content length: {np.mean(content_lengths):.0f} chars")
print(f"Samples with system prompt: {has_system}/{len(raw_dataset)} ({has_system/len(raw_dataset)*100:.1f}%)")
print(f"Samples with assistant: {has_assistant}/{len(raw_dataset)} ({has_assistant/len(raw_dataset)*100:.1f}%)")

# Turn 分布
turn_counter = Counter(turn_counts)
print(f"\nTurn distribution:")
for k, v in sorted(turn_counter.items()):
    print(f"  {k} turns: {v} samples")

In [ ]:
# ============================================================
# 3. 数据清洗与过滤
# ============================================================

def is_valid_sample(sample: dict) -> bool:
    """
    校验样本质量:
    - 必须有 assistant 回复
    - assistant 内容不能为空或过短
    - 总长度不能过长
    """
    msgs = sample.get("messages", [])
    if len(msgs) < 2:
        return False

    # 找到 assistant 消息
    assistant_msgs = [m for m in msgs if m.get("role") == "assistant"]
    if not assistant_msgs:
        return False

    # assistant 回复至少 20 字符
    for am in assistant_msgs:
        if len(am.get("content", "").strip()) < 20:
            return False

    # 总长度不超过 100K 字符 (避免超长样本)
    total_len = sum(len(m.get("content", "")) for m in msgs)
    if total_len > 100_000:
        return False

    return True

# 过滤
print(" Filtering dataset...")
filtered_dataset = raw_dataset.filter(is_valid_sample, desc="Validating samples")
print(f"  Before: {len(raw_dataset)} → After: {len(filtered_dataset)}")
print(f"  Removed: {len(raw_dataset) - len(filtered_dataset)} samples ({(1 - len(filtered_dataset)/len(raw_dataset))*100:.1f}%)")

In [ ]:
# ============================================================
# 4. 划分训练/验证集
# ============================================================

TRAIN_RATIO = 0.95

# Shuffle
filtered_dataset = filtered_dataset.shuffle(seed=SEED)

# Split
split = filtered_dataset.train_test_split(
    test_size=1 - TRAIN_RATIO,
    seed=SEED
)

split_dataset = DatasetDict({
    "train": split["train"],
    "validation": split["test"],
})

print(f" Train: {len(split_dataset['train'])} samples")
print(f" Validation: {len(split_dataset['validation'])} samples")

In [ ]:
# ============================================================
# 5. 格式化为 ChatML (SFT 训练用)
# ============================================================

sft_formatter = SFTDataFormatter(max_seq_length=2048)

# 应用格式化
print(" Formatting to ChatML...")

def format_sft(example):
    return {"text": sft_formatter.format(example)}

sft_dataset = split_dataset.map(
    format_sft,
    remove_columns=split_dataset["train"].column_names,
    desc="Formatting SFT data"
)

# 保存为 HuggingFace Dataset (Arrow 格式)
SFT_DATA_DIR = PROJECT_ROOT / "data" / "sft_formatted"
sft_dataset.save_to_disk(str(SFT_DATA_DIR))
print(f" SFT data saved to: {SFT_DATA_DIR}")

# 预览一条
print(f"\n=== Preview (train[0]) ===")
print(sft_dataset["train"][0]["text"][:400])
print("...")

In [ ]:
# ============================================================
# 6. 生成 RL 训练数据 (高质量的 prompt+answer 对)
# ============================================================

rl_formatter = RLDataFormatter(max_prompt_length=1536)

# RL 阶段通常只需 1000-2000 条高质量样本
RL_MAX_SAMPLES = 2000

# 筛选: 优先选择有明确答案标签的样本
def has_answer_tag(sample):
    """筛选包含 <answer> 标签的样本（适合 RL reward 计算）"""
    for msg in sample.get("messages", []):
        if msg.get("role") == "assistant":
            content = msg.get("content", "")
            if "<answer>" in content.lower() and "</answer>" in content.lower():
                return True
    return False

print(" Filtering for RL (samples with <answer> tags)...")
rl_candidates = split_dataset["train"].filter(has_answer_tag, desc="Filtering RL candidates")
print(f"  Candidates: {len(rl_candidates)} samples")

# 打乱取前 N 条
rl_candidates = rl_candidates.shuffle(seed=SEED).select(range(min(RL_MAX_SAMPLES, len(rl_candidates))))

# 格式化为 (prompt, answer) 对
print("\n Formatting RL data...")

def format_rl(example):
    sample = rl_formatter.format(example)
    return {
        "prompt": sample.prompt,
        "answer": sample.answer,
    }

rl_dataset = rl_candidates.map(format_rl, desc="Formatting RL data")

# 保存为 JSONL
RL_DATA_DIR = PROJECT_ROOT / "data"
RL_DATA_DIR.mkdir(parents=True, exist_ok=True)

rl_output_path = RL_DATA_DIR / "rl_train.jsonl"
with open(rl_output_path, "w", encoding="utf-8") as f:
    for sample in rl_dataset:
        f.write(json.dumps(sample, ensure_ascii=False) + "\n")

print(f" RL data saved to: {rl_output_path}")
print(f" RL samples: {len(rl_dataset)}")

# 预览
print(f"\n=== RL Sample 0 ===")
s = rl_dataset[0]
print(f"Prompt: {s['prompt'][:200]}...")
print(f"Answer: {s['answer'][:200]}...")

In [ ]:
# ============================================================
# 7. 数据质量报告
# ============================================================

print("=" * 60)
print("  DATA PREPARATION SUMMARY")
print("=" * 60)
print(f"\n  Input datasets:")
for ds in DATASETS:
    print(f"    - {ds}")
print(f"\n  Raw samples:     {len(raw_dataset):>8,}")
print(f"  After filtering: {len(filtered_dataset):>8,}")
print(f"\n  SFT Train:       {len(sft_dataset['train']):>8,} samples")
print(f"  SFT Validation:   {len(sft_dataset['validation']):>8,} samples")
print(f"\n  RL Train:        {len(rl_dataset):>8,} samples")
print(f"\n  Output files:")
print(f"    SFT: {SFT_DATA_DIR}/")
print(f"    RL:  {rl_output_path}")
print("\n" + "=" * 60)
print("  ✅ Data preparation complete! Proceed to Notebook 2.")
print("=" * 60)